# Schubert diagnostics for incomplete asset markets

Companion notebook to *A note on Schubert calculus in heterogeneous pure exchange economies with incomplete markets*.

Given a payoff matrix `A` (states × assets) whose rows are **ordered from the worst state to the best state**, the notebook computes:

1. the lower- and upper-tail insurability profiles `d`, `d~`;
2. the Arrow skeleton (pivot states), the partition λ and the excess tail insurability `Exc`;
3. the reduced basis (Schubert-cell coordinates);
4. the effect of a financial innovation and the greedy optimal tail innovation;
5. a full-insurance test and the residual (uninsurable) risk under mean–variance preferences.

Every function works with floating-point data through a numerical rank tolerance.

In [1]:
import numpy as np
from itertools import combinations

TOL = 1e-9

def rank(M, tol=TOL):
    M = np.atleast_2d(np.asarray(M, dtype=float))
    if M.size == 0:
        return 0
    return int(np.linalg.matrix_rank(M, tol=tol))

def profile(A, tol=TOL):
    """Lower-tail profile d_i = dim(M ∩ F_i) = J - rank(A_{>i}), i = 0..S."""
    A = np.asarray(A, dtype=float); S, J = A.shape
    return [J - rank(A[i:, :], tol) if i < S else J for i in range(S + 1)]

def upper_profile(A, tol=TOL):
    """Upper-tail profile d~_b = dim(M ∩ F~_b) = J - rank(A_{<= S-b}), b = 0..S."""
    A = np.asarray(A, dtype=float); S, J = A.shape
    return [J - rank(A[:S - b, :], tol) if b < S else J for b in range(S + 1)]

def pivots(d):
    return [i for i in range(1, len(d)) if d[i] - d[i - 1] == 1]

def partition(p, S, J):
    n = S - J
    return [n + k + 1 - p[k] for k in range(J)]   # k is 0-based here

def excess(d, S, J):
    n = S - J
    return sum(d[i] - max(0, i - n) for i in range(1, S + 1))

def diagnostics(A, name=""):
    A = np.asarray(A, dtype=float); S, J = A.shape
    assert rank(A) == J, "redundant assets: drop them first"
    d = profile(A); dt = upper_profile(A); p = pivots(d)
    lam = partition(p, S, J); pt = pivots(dt); mu = partition(pt, S, J)
    out = dict(name=name, S=S, J=J, profile=d, skeleton=p, partition=lam,
               excess=excess(d, S, J), upper_profile=dt, upper_partition=mu,
               upper_excess=excess(dt, S, J))
    return out

def show(D):
    print(f"{D['name']}:  S={D['S']}, J={D['J']}")
    print(f"  lower profile d   = {D['profile']}")
    print(f"  Arrow skeleton    = {D['skeleton']}   partition λ = {D['partition']}   Exc = {D['excess']}")
    print(f"  upper profile d~  = {D['upper_profile']}   partition μ = {D['upper_partition']}   Exc~ = {D['upper_excess']}")


## Reduced basis (Schubert-cell coordinates)

The reduced basis is obtained by Gaussian elimination on the columns of `A`, using as pivot of each vector its **last** non-zero coordinate.

In [2]:
def reduced_basis(A, tol=TOL):
    A = np.asarray(A, dtype=float); S, J = A.shape
    d = profile(A, tol); p = pivots(d)
    # basis vectors as rows; eliminate from the bottom (worst-to-best ordering => last nonzero)
    U = A.T.copy()
    piv = []
    rows = list(range(J))
    for col in reversed(range(S)):
        cand = [r for r in rows if abs(U[r, col]) > tol]
        if not cand:
            continue
        r = cand[0]
        U[r] /= U[r, col]
        for q in range(J):
            if q != r and abs(U[q, col]) > tol:
                U[q] -= U[q, col] * U[r]
        piv.append((col + 1, r)); rows.remove(r)
    order = [r for _, r in sorted(piv)]
    U = U[order]
    U[np.abs(U) < tol] = 0.0
    assert sorted(c for c, _ in piv) == p
    return U, p

def gale_max_minor(A, tol=TOL):
    A = np.asarray(A, dtype=float); S, J = A.shape
    nz = [c for c in combinations(range(1, S + 1), J)
          if abs(np.linalg.det(A[[x - 1 for x in c], :])) > tol]
    best = [c for c in nz if all(all(t[k] <= c[k] for k in range(J)) for t in nz)]
    return best[0] if best else None


## Financial innovation

`innovation(A, y)` returns the new pivot τ and the change in excess (Theorem: `Exc' - Exc = n - τ`).
`greedy_tail_innovation(A, k)` adds Arrow securities on the k worst non-pivot states, which is componentwise optimal.

In [3]:
def innovation(A, y):
    A = np.asarray(A, dtype=float); S, J = A.shape
    A2 = np.column_stack([A, y])
    assert rank(A2) == J + 1, "the new asset is redundant"
    p, p2 = pivots(profile(A)), pivots(profile(A2))
    tau = sorted(set(p2) - set(p))[0]
    dE = excess(profile(A2), S, J + 1) - excess(profile(A), S, J)
    assert dE == (S - J) - tau
    return tau, dE

def greedy_tail_innovation(A, k):
    A = np.asarray(A, dtype=float); S, J = A.shape
    p = set(pivots(profile(A)))
    new = [s for s in range(1, S + 1) if s not in p][:k]
    E = np.eye(S)[:, [s - 1 for s in new]]
    return np.column_stack([A, E]), new


## Equilibrium: full insurance test and residual risk

`full_insurance(A, eps)` checks whether every endowment risk lies in the market span (Theorem on full insurance).
`residual_risk(A, eps, pi)` returns the equilibrium uninsured component `(I - P)ε` under mean–variance preferences, where `P` is the `L²(π)`-orthogonal projection onto the market span.

In [4]:
def in_span(A, v, tol=TOL):
    return rank(np.column_stack([A, v]), tol) == rank(A, tol)

def full_insurance(A, eps):
    return all(in_span(A, e) for e in eps)

def residual_risk(A, eps, pi):
    A = np.asarray(A, dtype=float); W = np.diag(pi)
    P = A @ np.linalg.solve(A.T @ W @ A, A.T @ W)          # L2(pi) projection onto span(A)
    res = [(np.eye(len(pi)) - P) @ e for e in eps]
    var = [float(r @ W @ r) for r in res]                   # E_pi[r^2] (mean zero)
    return res, var


## Example 1 — options and tail insurance

Six states ordered by the payoff of a stock, from worst to best. Assets: a bond, the stock, a put with strike 3 and a call with strike 4.

In [5]:
s    = np.arange(1, 7, dtype=float)          # stock payoff in states 1..6 (worst -> best)
bond = np.ones(6)
put  = np.maximum(3 - s, 0)                     # (2,1,0,0,0,0)
call = np.maximum(s - 4, 0)                     # (0,0,0,0,1,2)

markets = {
    "bond + stock":               np.column_stack([bond, s]),
    "bond + stock + put(3)":      np.column_stack([bond, s, put]),
    "bond + stock + call(4)":     np.column_stack([bond, s, call]),
    "bond + stock + put + call":  np.column_stack([bond, s, put, call]),
}
for name, A in markets.items():
    show(diagnostics(A, name))
    print("  Gale-max non-vanishing minor:", gale_max_minor(A))


bond + stock:  S=6, J=2
  lower profile d   = [0, 0, 0, 0, 0, 1, 2]
  Arrow skeleton    = [5, 6]   partition λ = [0, 0]   Exc = 0
  upper profile d~  = [0, 0, 0, 0, 0, 1, 2]   partition μ = [0, 0]   Exc~ = 0
  Gale-max non-vanishing minor: (5, 6)
bond + stock + put(3):  S=6, J=3
  lower profile d   = [0, 0, 1, 1, 1, 2, 3]
  Arrow skeleton    = [2, 5, 6]   partition λ = [2, 0, 0]   Exc = 2
  upper profile d~  = [0, 0, 0, 1, 1, 2, 3]   partition μ = [1, 0, 0]   Exc~ = 1
  Gale-max non-vanishing minor: (2, 5, 6)
bond + stock + call(4):  S=6, J=3
  lower profile d   = [0, 0, 0, 1, 1, 2, 3]
  Arrow skeleton    = [3, 5, 6]   partition λ = [1, 0, 0]   Exc = 1
  upper profile d~  = [0, 0, 1, 1, 1, 2, 3]   partition μ = [2, 0, 0]   Exc~ = 2
  Gale-max non-vanishing minor: (3, 5, 6)
bond + stock + put + call:  S=6, J=4
  lower profile d   = [0, 0, 1, 2, 2, 3, 4]
  Arrow skeleton    = [2, 3, 5, 6]   partition λ = [1, 1, 0, 0]   Exc = 2
  upper profile d~  = [0, 0, 1, 2, 2, 3, 4]   partition μ = [

In [6]:
A = markets["bond + stock"]
print("adding put(3):  tau, ΔExc =", innovation(A, put))
print("adding call(4): tau, ΔExc =", innovation(A, call))
A_g, new = greedy_tail_innovation(A, 1)
print("greedy tail innovation adds Arrow security on state", new)
show(diagnostics(A_g, "bond + stock + greedy Arrow"))


adding put(3):  tau, ΔExc = (2, 2)
adding call(4): tau, ΔExc = (3, 1)
greedy tail innovation adds Arrow security on state [1]
bond + stock + greedy Arrow:  S=6, J=3
  lower profile d   = [0, 1, 1, 1, 1, 2, 3]
  Arrow skeleton    = [1, 5, 6]   partition λ = [3, 0, 0]   Exc = 3
  upper profile d~  = [0, 0, 0, 0, 1, 2, 3]   partition μ = [0, 0, 0]   Exc~ = 0


## Example 2 — insuring an idiosyncratic tail risk

Four equally likely states. Two agents bear opposite risks ±(1, −1, 0, 0), concentrated in the two worst states.

In [7]:
pi  = np.full(4, 0.25)
eps = [np.array([1., -1., 0., 0.]), np.array([-1., 1., 0., 0.])]
one = np.ones(4)
cases = {
    "M1 = <1, (0,0,1,1)>":              np.column_stack([one, [0, 0, 1, 1]]),
    "M2 = <1, (1,-1,0,0)>":             np.column_stack([one, [1, -1, 0, 0]]),
    "M3 = M1 + Arrow(state 1)":         np.column_stack([one, [0, 0, 1, 1], [1, 0, 0, 0]]),
    "M4 = <1, (1,2,3,4)>  (generic)":   np.column_stack([one, [1, 2, 3, 4]]),
}
for name, A in cases.items():
    D = diagnostics(A, name)
    _, var = residual_risk(A, eps, pi)
    print(f"{name:34s} skeleton={D['skeleton']} Exc={D['excess']} full insurance={full_insurance(A, eps)} "
          f"residual variance={var[0]:.4f}")


M1 = <1, (0,0,1,1)>                skeleton=[2, 4] Exc=1 full insurance=False residual variance=0.5000
M2 = <1, (1,-1,0,0)>               skeleton=[2, 4] Exc=1 full insurance=True residual variance=0.0000
M3 = M1 + Arrow(state 1)           skeleton=[1, 2, 4] Exc=2 full insurance=True residual variance=0.0000
M4 = <1, (1,2,3,4)>  (generic)     skeleton=[3, 4] Exc=0 full insurance=False residual variance=0.4500


## Example 3 — genericity check by simulation

Random Gaussian payoff matrices almost surely have zero excess tail insurability.

In [8]:
rng = np.random.default_rng(0)
S, J, N = 8, 3, 2000
exc = [diagnostics(rng.standard_normal((S, J)))['excess'] for _ in range(N)]
print(f"share of random markets with Exc = 0: {np.mean(np.array(exc) == 0):.3f}")


share of random markets with Exc = 0: 1.000


## Guaranteed average insurance (Proposition `prop:average`)
For an isotropic random claim in $F_i$, the expected insured share is $\operatorname{tr}(QPQ)/i \ge d_i(M)/i$, with equality iff $d_i+\tilde d_{S-i}=J$.

In [9]:
import numpy as np
def average_share(A, pi, i):
    """Expected insured share of an isotropic random claim in F_i, and the guarantee d_i/i."""
    A = np.asarray(A, float); pi = np.asarray(pi, float)
    S, J = A.shape
    Pi = np.diag(pi)
    P = A @ np.linalg.solve(A.T @ Pi @ A, A.T @ Pi)
    B = np.diag(1/np.sqrt(pi))[:, :i]                 # pi-orthonormal basis of F_i
    share = sum(B[:, s] @ Pi @ P @ B[:, s] for s in range(i)) / i
    d_i = J - np.linalg.matrix_rank(A[i:]) if i < S else J
    return share, d_i / i

one = np.ones(4)
markets = {
    "M1": np.column_stack([one, [0,0,1,1]]),
    "M2": np.column_stack([one, [1,-1,0,0]]),
    "M3": np.column_stack([one, [0,0,1,1], [1,0,0,0]]),
    "M4": np.column_stack([one, [1,2,3,4]]),
}
pi = np.ones(4)/4
for name, A in markets.items():
    share, bound = average_share(A, pi, 2)
    print(f"{name}: expected insured share = {share:.3f},  guarantee d_2/2 = {bound:.3f}")

M1: expected insured share = 0.500,  guarantee d_2/2 = 0.500
M2: expected insured share = 0.750,  guarantee d_2/2 = 0.500
M3: expected insured share = 1.000,  guarantee d_2/2 = 1.000
M4: expected insured share = 0.500,  guarantee d_2/2 = 0.000
